# 13 · Powerlifting：其他模型大比较（Colab GPU，五折）

**目的**：回答「别的模型能不能用」。所有模型用**同样的最终特征**（11 中 LightGBM / TabM 用的 R2 + H2 + E，57 列）、**同样的五折**（种子 42）和同样的内层划分，比较两件事：
1. **单独的分数**：和 11 中各模型种子 42 的分数直接可比（单种子对单种子）。
2. **能不能帮到最终融合**：加到 11 的 6 模型融合里，五折嵌套 NNLS 的分数变化和权重。融合只看「错误是否不同」，单独弱的模型也可能有用。

|模型|类型|说明|
|---|---|---|
|`ridge`|线性|数值中位数填补 + 标准化，类别 one-hot（稀有类别合并），Ridge|
|`hgb`|梯度提升树（sklearn）|HistGradientBoosting，原生类别（最多 250 类，其余合并）|
|`et`|随机化树集成|ExtraTrees，200 棵|
|`rf`|随机森林|RandomForest，200 棵，每棵用一半样本|
|`tabpfn`|表格基础模型|TabPFN：默认 v2（无需登录，上下文 10,000 行）；在 Colab「密钥」里设置 `TABPFN_TOKEN` 时用最新版|
|`mlp_plr`|神经网络|MLP + PLR 数值嵌入（pytabkit）|
|`resnet`|神经网络|ResNet（pytabkit）|
|`ebm`|可解释提升（GA2M）|InterpretML 的 EBM，可画出每个特征的形状函数，适合写进报告|
|`ftt`、`tabr`|Transformer / 检索式网络|默认不跑：FT-Transformer 很慢；TabR 需要 faiss|

已经比较过、这里不再重复的：v1/v2 的自写 MLP（单种子 80.05）、RealMLP（78.48，每折 15 分钟）、决策树（88.5）、早期 sklearn GB（82.6）。

**判读**：加入后嵌套 NNLS 平均 ΔRMSE ≤ −0.02 且 ≥ 4/5 折更好，才在最后写一个加入新模型的提交文件。

**操作**：Colab 选 **T4 GPU**，从上到下运行；约 1.5 小时（EBM、两个神经网络、TabPFN 较慢）。某个模型安装或运行失败会自动跳过，不影响其他模型。按模型逐个跑完，中途中断后从第 8 节开始运行可以用已经跑完的模型。需要 11 的结果目录 `artifacts/final_v1/`。

## 0. 安装额外的包（在导入 numpy 之前）

In [1]:
import importlib
import importlib.metadata
import subprocess
import sys

EXTRA_PACKAGES = ['interpret-core', 'tabpfn']        # EBM、TabPFN
missing = []
for package in EXTRA_PACKAGES:
    try:
        importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        missing.append(package)
if missing:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], text=True,
                            capture_output=True, check=False)
    print('pip 安装：', missing, '返回码', result.returncode)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    importlib.invalidate_caches()

pip 安装： ['interpret-core', 'tabpfn'] 返回码 0


## 1. 环境与配置（与 11 相同，实验名不同）

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import inspect
import json
import os
import shutil
import subprocess
import sys
import time
import warnings

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

PROJECT_DIR = Path(os.environ.get(
    'PL_PROJECT_DIR',
    '/content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting',
)).expanduser().resolve()
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'找不到项目目录：{PROJECT_DIR}；请修改 PROJECT_DIR')

CATBOOST_VERSION = '1.2.10'
to_install = []
try:
    installed_catboost = importlib.metadata.version('catboost')
except importlib.metadata.PackageNotFoundError:
    installed_catboost = None
if installed_catboost != CATBOOST_VERSION:
    if 'catboost' in sys.modules:
        raise RuntimeError('当前运行时已经导入另一版 CatBoost；请重启运行时后从头运行')
    to_install.append(f'catboost=={CATBOOST_VERSION}')
try:
    importlib.metadata.version('pytabkit')
except importlib.metadata.PackageNotFoundError:
    to_install.append('pytabkit')
if to_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *to_install])
    importlib.invalidate_caches()

import catboost
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from IPython.display import display
from scipy.optimize import nnls
from sklearn.model_selection import KFold, train_test_split

try:
    from pytabkit import TabM_D_Regressor
    PYTABKIT_ERROR = None
except Exception as exc:                       # 安装或导入失败时其余模型照常运行
    TabM_D_Regressor = None
    PYTABKIT_ERROR = repr(exc)

warnings.filterwarnings('ignore', category=UserWarning)
if catboost.__version__ != CATBOOST_VERSION:
    raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}；实际为 {catboost.__version__}')
if shutil.which('nvidia-smi') is None or not torch.cuda.is_available():
    raise RuntimeError('没有可用的 GPU：请在 Colab 选择 T4 GPU 后从头运行')
GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                          text=True, capture_output=True, check=False).stdout.strip().splitlines()[0]

EXPERIMENT_ID = 'model_compare_v1'
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / EXPERIMENT_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42                                  # 07–10 的五折划分种子（第一个种子）
SEEDS = [42, 7, 2024]                      # 每个种子：不同的五折划分、内层早停划分和模型随机性
N_FOLDS = 5
INNER_STOP_FRACTION = 0.10
MAX_ROUNDS = 20000
FINAL_MODELS = ['lgb', 'xgb', 'lgb_rel', 'catboost', 'cb_rel', 'tabm']   # 每折内的运行顺序：快的先跑
FULL_REFIT = True                          # 树模型在全部训练行上重训
REFIT_FACTOR = 1.2                         # 全量重训轮数 = CV 平均最佳轮数 × 1.2（数据多约 39%）
RUN_CLEANING_TEST = True                   # 第 9 节：数据清洗实验（只作报告证据）
MODELS = ['lgb', 'xgb', 'catboost']        # 09 / 10 特征单元使用的三个模型族名
RUN_ONLY = None                            # 09 / 10 特征单元需要的变量，这里不用
KEEP_DELTA = -0.05
EXPECTED_SHA256 = {'train.csv': '62319814116506fb95a51f1185dbacdb170ab055642ee4c347a30584093d0856', 'test.csv': '563c6fa29f9e535740ef1f09b25954f00fbe242c81c0e5a7343dc18aa962ba46'}
VERSIONS = {'catboost': catboost.__version__, 'xgboost': xgb.__version__, 'lightgbm': lgb.__version__,
            'torch': torch.__version__, 'pytabkit': None if PYTABKIT_ERROR else importlib.metadata.version('pytabkit'),
            'numpy': np.__version__, 'pandas': pd.__version__, 'python': sys.version.split()[0]}
print('PROJECT_DIR :', PROJECT_DIR)
print('OUTPUT_DIR  :', OUTPUT_DIR)
print('GPU         :', GPU_NAME)
print('版本         :', VERSIONS)
if PYTABKIT_ERROR:
    print('pytabkit 不可用，TabM 会在预检中跳过：', PYTABKIT_ERROR)

PREV_DIR = PROJECT_DIR / 'artifacts' / 'final_v1'          # 11 的结果
BOOST_DIR = PROJECT_DIR / 'artifacts' / 'final_v2'         # 12 的结果（如已运行）
PREV_SEEDS = [42, 7, 2024]
COMPARE = ['ridge', 'hgb', 'et', 'rf', 'tabpfn', 'mlp_plr', 'resnet', 'ebm']   # 运行顺序：快的先跑
OPTIONAL = []                         # 想多试可以加 'ftt'（很慢）、'tabr'（需要 faiss）
TABPFN_CONTEXT_ROWS = 10000           # TabPFN v2 的预训练上限是 10,000 行
ADD_DELTA = -0.02                     # 加入融合的门槛：嵌套 ΔRMSE ≤ −0.02 且 ≥ 4/5 折
TABPFN_TOKEN = os.environ.get('TABPFN_TOKEN')
if not TABPFN_TOKEN and drive is not None:
    try:
        from google.colab import userdata
        TABPFN_TOKEN = userdata.get('TABPFN_TOKEN')
    except Exception:
        TABPFN_TOKEN = None
if TABPFN_TOKEN:
    os.environ['TABPFN_TOKEN'] = TABPFN_TOKEN
print('TabPFN：', '最新版（已提供 TABPFN_TOKEN）' if TABPFN_TOKEN else 'v2（无需登录）')
if not (PREV_DIR / 'summary_weights.csv').is_file():
    raise FileNotFoundError(f'缺少 11 的结果：{PREV_DIR}；请先完整运行 11')

Mounted at /content/drive
PROJECT_DIR : /content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting
OUTPUT_DIR  : /content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting/artifacts/model_compare_v1
GPU         : Tesla T4, 15360 MiB
版本         : {'catboost': '1.2.10', 'xgboost': '3.4.1', 'lightgbm': '4.6.0', 'torch': '2.11.0+cu130', 'pytabkit': '1.7.3', 'numpy': '2.1.3', 'pandas': '2.2.3', 'python': '3.13.15'}
TabPFN： v2（无需登录）


## 2. 读取并核对数据（与 07 相同）

In [3]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def rmse(a, b):
    a, b = np.asarray(a, dtype='float64'), np.asarray(b, dtype='float64')
    return float(np.sqrt(np.mean((a - b) ** 2)))


for name, expected in EXPECTED_SHA256.items():
    require(sha256_file(PROJECT_DIR / name) == expected, f'{name} 与本地版本不同；请检查 Drive 上的数据')

TARGET = 'TotalKg'
train = pd.read_csv(PROJECT_DIR / 'train.csv')
test = pd.read_csv(PROJECT_DIR / 'test.csv')
RAW_NUM = ['Age', 'BodyweightKg', 'MeetYear']
RAW_FEATURES = [c for c in test.columns if c != 'id']
CAT = [c for c in RAW_FEATURES if c not in RAW_NUM]          # 不依赖 object / str dtype（pandas 2 与 3 都可用）
require(train.shape == (105000, 18) and test.shape == (45000, 17), '数据形状不符')
require(len(CAT) == 13 and all(pd.api.types.is_numeric_dtype(train[c]) for c in RAW_NUM)
        and not any(pd.api.types.is_numeric_dtype(train[c]) for c in CAT), '列分组不符')
require(not train[RAW_FEATURES].isna().any().any() and not test[RAW_FEATURES].isna().any().any(), '出现空值')
y = train[TARGET].to_numpy(dtype='float64')
n_train = len(train)

FOLD_IDS = np.zeros(n_train, dtype=int)
for k, (_, valid) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.arange(n_train)), start=1):
    FOLD_IDS[valid] = k
print(f'train {train.shape}  test {test.shape}；每折验证行数：{np.bincount(FOLD_IDS)[1:].tolist()}')

train (105000, 18)  test (45000, 17)；每折验证行数：[21000, 21000, 21000, 21000, 21000]


## 3. 第一轮特征（与 07 相同）

In [4]:
both = pd.concat([train[RAW_FEATURES], test[RAW_FEATURES]], ignore_index=True)

# F0：v2 的体重级别特征
weight_class = pd.to_numeric(both['WeightClassKg'].str.replace('+', '', regex=False), errors='coerce')
F0_EXTRA = pd.DataFrame({
    'WeightClass_num': weight_class,
    'Is_SHW': both['WeightClassKg'].str.endswith('+').astype('float64'),
    'BW_minus_Class': both['BodyweightKg'] - weight_class,
})

# P：记录精度（树按阈值切数值，学不到「是不是 .5」「是不是整数」）
age2 = (both['Age'] * 2).round()
require(np.allclose(both['Age'] * 2, age2), 'Age 不是 0.5 的整数倍')
bw = both['BodyweightKg']
is_int = np.isclose(bw, bw.round())
is_1dec = np.isclose(bw * 10, (bw * 10).round())
is_half = is_1dec & ~is_int & np.isclose((bw * 10).round() % 10, 5)
pounds = bw / 0.45359237
P = pd.DataFrame({
    'age_half': (age2 % 2).astype('float64'),
    'bw_precision': np.select([is_int, is_half, is_1dec], [0.0, 1.0, 2.0], default=3.0),
    'bw_lb_int': (np.abs(pounds - pounds.round()) < 0.015).astype('float64'),
})

# T：跨地参赛（居住地未知时为 NaN）
abroad = (both['Country'] != both['MeetCountry']).astype('float64').where(both['Country'] != 'Unknown')
state_known = ((both['State'] != 'Unknown') & (both['MeetState'] != 'Unknown')
               & (both['Country'] == both['MeetCountry']))
other_state = (both['State'] != both['MeetState']).astype('float64').where(state_known)
T = pd.DataFrame({'abroad': abroad, 'other_state': other_state})

# M：比赛背景；「比赛」≈ 联盟 × 年份 × 比赛国 × 比赛州（train+test 无标签统计）
meet = (both['Federation'] + '|' + both['MeetYear'].astype(str) + '|'
        + both['MeetCountry'] + '|' + both['MeetState'])


def by_meet(values):
    return values.groupby(meet).transform('mean')


M = pd.DataFrame({
    'meet_n': meet.map(meet.value_counts()).astype('float64'),
    'meet_sexeq_n': both.groupby([meet, both['Sex'], both['Equipment']])['Age'].transform('size').astype('float64'),
    'fed_year_n': both.groupby(['Federation', 'MeetYear'])['Age'].transform('size').astype('float64'),
    'meet_share_other_state': by_meet(other_state),
    'meet_share_abroad': by_meet(abroad),
    'meet_share_age_half': by_meet(P['age_half']),
    'meet_share_equipped': by_meet((both['Equipment'] != 'Raw').astype('float64')),
    'meet_share_male': by_meet((both['Sex'] == 'M').astype('float64')),
    'meet_mean_age': by_meet(both['Age']),
})

BLOCKS = {'P': P, 'T': T, 'M': M}
VARIANTS = {'F0': [], 'P': ['P'], 'T': ['T'], 'M': ['M'], 'PT': ['P', 'T'], 'ALL': ['P', 'T', 'M']}
for name, frame in {'F0': F0_EXTRA, **BLOCKS}.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')

_FRAMES = {}


def frames_for(variant, model):
    """(train, test) 特征表。CatBoost 的类别为字符串；LGB / XGB 为 category（词表来自 train+test，无标签）。"""
    kind = 'str' if model == 'catboost' else 'category'
    if (variant, kind) not in _FRAMES:
        frame = pd.concat([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in VARIANTS[variant]] + [both[CAT]], axis=1)
        require(frame.columns.is_unique, f'{variant} 列名重复')
        frame[CAT] = frame[CAT].astype(str).astype(object) if kind == 'str' else frame[CAT].astype('category')
        _FRAMES[(variant, kind)] = (frame.iloc[:n_train].reset_index(drop=True),
                                    frame.iloc[n_train:].reset_index(drop=True))
    return _FRAMES[(variant, kind)]


print({v: frames_for(v, 'lgb')[0].shape[1] for v in VARIANTS})
summary_new = pd.concat([P, T, M], axis=1)
display(pd.DataFrame({'train_mean': summary_new.iloc[:n_train].mean(), 'test_mean': summary_new.iloc[n_train:].mean(),
                      'nan_share': summary_new.isna().mean()}).round(4))

{'F0': 19, 'P': 22, 'T': 21, 'M': 28, 'PT': 24, 'ALL': 33}


,train_mean,test_mean,nan_share
age_half,0.4817,0.4782,0.0000
bw_precision,1.9668,1.9609,0.0000
bw_lb_int,0.0500,0.0503,0.0000
abroad,0.0824,0.0822,0.0184
other_state,0.2670,0.2724,0.6491
meet_n,63.6623,63.9177,0.0000
meet_sexeq_n,33.0578,33.1368,0.0000
fed_year_n,653.6460,656.8197,0.0000
meet_share_other_state,0.2661,0.2673,0.6219
meet_share_abroad,0.0833,0.0837,0.0004


## 4. 第二轮特征（与 09 相同）

In [5]:
_FRAMES.clear()                     # 07 的变体表不再需要，释放内存
CORE9 = ['Age', 'BodyweightKg', 'MeetYear', 'Sex', 'Federation', 'MeetCountry', 'MeetState', 'Country', 'State']
sex, equipment = both['Sex'], both['Equipment']
state = both['State'].where(both['State'] != 'Unknown')
country = both['Country'].where(both['Country'] != 'Unknown')
national = both['Federation'] + '|' + both['MeetYear'].astype(str) + '|' + both['MeetCountry']


def counts(key):
    return key.map(key.value_counts()).astype('float64')


def n_distinct(values, key):
    return values.groupby(key).transform('nunique').astype('float64')


# M2：比赛背景加强（meet 与 by_meet 来自 07 的特征单元）
M2 = pd.DataFrame({
    'meet_n_states': n_distinct(state, meet),
    'meet_n_countries': n_distinct(country, meet),
    'meet_n_wclass': n_distinct(both['WeightClassKg'], meet),
    'meet_share_tested': by_meet((both['Tested'] == 'Yes').astype('float64')),
    'meet_share_raw': by_meet((equipment == 'Raw').astype('float64')),
    'meet_share_multi': by_meet((equipment == 'Multi-ply').astype('float64')),
    'meet_share_bw2dec': by_meet((P['bw_precision'] == 3).astype('float64')),
    'meet_share_junior': by_meet((both['Age'] < 24).astype('float64')),
    'meet_share_master': by_meet((both['Age'] >= 40).astype('float64')),
    'meet_age_std': both['Age'].groupby(meet).transform('std'),
    'meet_sex_bw_mean': both['BodyweightKg'].groupby([meet, sex]).transform('mean'),
    'nat_n': counts(national),
    'nat_n_meetstates': n_distinct(both['MeetState'], national),
    'nat_n_states': n_distinct(state, national),
})

# D：组别背景
division = meet + '|' + sex + '|' + equipment + '|' + both['WeightClassKg']
age_division = meet + '|' + sex + '|' + equipment + '|' + both['AgeClass']
exact_key = pd.Series(pd.util.hash_pandas_object(both[RAW_FEATURES], index=False).to_numpy())
core_key = pd.Series(pd.util.hash_pandas_object(both[CORE9], index=False).to_numpy())
D = pd.DataFrame({
    'div_n': counts(division),
    'div_bw_rank': both['BodyweightKg'].groupby(division).rank(pct=True),
    'div_age_rank': both['Age'].groupby(division).rank(pct=True),
    'agediv_n': counts(age_division),
    'dup_exact_n': counts(exact_key) - 1,
    'dup_core_n': counts(core_key) - 1,
})

# H：近似同一运动员的历史（出生年 = floor(MeetYear − Age)：x.5 年龄时恰为出生年，整数年龄可能差 1）
valid = (both['Country'] != 'Unknown').to_numpy()
birth = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int).astype(str)
athlete = sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation'] + '|' + birth
athlete0 = sex + '|' + both['Country'] + '|' + both['State'] + '|' + birth
year = both['MeetYear'].astype('float64')
H = pd.DataFrame({
    'ath_n': counts(athlete),
    'ath_n_years': n_distinct(year, athlete),
    'ath_years_since_first': year - year.groupby(athlete).transform('min'),
    'ath_years_to_last': year.groupby(athlete).transform('max') - year,
    'ath_n_equipment': n_distinct(equipment, athlete),
    'ath_bw_delta': both['BodyweightKg'] - both['BodyweightKg'].groupby(athlete).transform('mean'),
    'ath0_n': counts(athlete0),
    'ath0_n_feds': n_distinct(both['Federation'], athlete0),
})
H.loc[~valid] = np.nan


# S：DOTS 系数（TotalKg = DOTS 分 × dots_scale）
def dots_scale(sex_values, bw_values):
    s = np.asarray(sex_values).astype(str)
    bw = np.asarray(bw_values, dtype='float64')
    m, f = np.clip(bw, 40, 210), np.clip(bw, 40, 150)
    poly_m = -1.0930e-6 * m**4 + 7.391293e-4 * m**3 - 0.1918759221 * m**2 + 24.0900756 * m - 307.75076
    poly_f = -1.0706e-6 * f**4 + 5.158568e-4 * f**3 - 0.1126655495 * f**2 + 13.6175032 * f - 57.96288
    return np.where(s == 'M', poly_m, np.where(s == 'F', poly_f, 0.5 * (poly_m + poly_f))) / 500.0


S = pd.DataFrame({'dots_scale': dots_scale(sex, both['BodyweightKg'])})

# C：只给 CatBoost 的高基数 ID（有序目标统计编码）
C = pd.DataFrame({'meet_id': meet, 'athlete_id': (athlete + '|' + equipment).where(valid, 'unknown')})

BLOCKS2 = {'M2': M2, 'D': D, 'H': H, 'S': S}
CB_ONLY = {'C': C}
for name, frame in {**BLOCKS2, **CB_ONLY}.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
for name, frame in BLOCKS2.items():
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
require(S['dots_scale'].min() > 0.3, 'DOTS 系数异常')

R1_BLOCKS = ['P', 'T', 'M']
VARIANT_BLOCKS = {'R1': [], 'M2': ['M2'], 'D': ['D'], 'H': ['H'], 'S': ['S'], 'C': ['C']}
MODELS_FOR = {'C': ['catboost']}      # C 只改变 CatBoost；融合时 LightGBM / XGBoost 用 R1
SCREEN = [v for v in VARIANT_BLOCKS if RUN_ONLY is None or v in RUN_ONLY or v == 'R1']
_FRAMES2 = {}


def models_for(variant):
    return MODELS_FOR.get(variant, MODELS)


def frames2(variant, model):
    """(train, test, 类别列)。R1 = F0 + P + T + M；C 块只加给 CatBoost。"""
    blocks = VARIANT_BLOCKS[variant]
    kind = 'str' if model == 'catboost' else 'category'
    key = (tuple(blocks), kind)
    if key not in _FRAMES2:
        nums = ([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in R1_BLOCKS]
                + [BLOCKS2[b] for b in blocks if b in BLOCKS2])
        cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
        frame = pd.concat(nums + cats, axis=1)
        require(frame.columns.is_unique, f'{variant} 列名重复')
        cat_cols = [c for part in cats for c in part.columns]
        if kind == 'str':
            frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
        else:
            frame[cat_cols] = frame[cat_cols].astype('category')
        _FRAMES2[key] = (frame.iloc[:n_train].reset_index(drop=True),
                         frame.iloc[n_train:].reset_index(drop=True), cat_cols)
    return _FRAMES2[key]


new_cols = pd.concat([M2, D, H, S], axis=1)
display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                      'nan_share': new_cols.isna().mean()}).round(4))
print('C 块类别数：', {c: int(C[c].nunique()) for c in C.columns})
print('各变体列数（LightGBM / XGBoost）：', {v: frames2(v, 'lgb')[0].shape[1] for v in VARIANT_BLOCKS if v != 'C'},
      '｜CatBoost C：', frames2('C', 'catboost')[0].shape[1])

,train_mean,test_mean,nan_share
meet_n_states,3.2039,3.2371,0.0000
meet_n_countries,2.0094,2.0170,0.0000
meet_n_wclass,10.7194,10.7360,0.0000
meet_share_tested,0.7627,0.7604,0.0000
meet_share_raw,0.6208,0.6212,0.0000
meet_share_multi,0.0234,0.0240,0.0000
meet_share_bw2dec,0.2949,0.2969,0.0000
meet_share_junior,0.4549,0.4538,0.0000
meet_share_master,0.1703,0.1714,0.0000
meet_age_std,10.0628,10.0618,0.0104


C 块类别数： {'meet_id': 10546, 'athlete_id': 46893}
各变体列数（LightGBM / XGBoost）： {'R1': 33, 'M2': 47, 'D': 39, 'H': 41, 'S': 34} ｜CatBoost C： 35


## 5. 第三轮特征与各模型的最终特征表（与 11 相同）

In [6]:
_FRAMES2.clear()                    # 09 的变体表不再需要，释放内存
birth_num = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int)


def fuzzy_count(prefix_key):
    """同前缀、出生年为 b−1、b、b+1 的行数（整数年龄推算的出生年可能差 1）。"""
    exact = prefix_key + '|' + birth_num.astype(str)
    table = exact.value_counts()
    total = exact.map(table).astype('float64')
    for d in (-1, 1):
        total = total + (prefix_key + '|' + (birth_num + d).astype(str)).map(table).fillna(0)
    return total


# H2：运动员生涯（athlete、valid 来自 09 的特征单元；M、T 来自 07；M2 来自 09）
bw = both['BodyweightKg']
H2 = pd.DataFrame({
    'ath_max_meet_states': M2['meet_n_states'].groupby(athlete).transform('max'),
    'ath_mean_meet_states': M2['meet_n_states'].groupby(athlete).transform('mean'),
    'ath_max_meet_n': M['meet_n'].groupby(athlete).transform('max'),
    'ath_max_nat_states': M2['nat_n_states'].groupby(athlete).transform('max'),
    'ath_ever_abroad': T['abroad'].groupby(athlete).transform('max'),
    'ath_n_wclass': n_distinct(both['WeightClassKg'], athlete),
    'ath_bw_range': bw.groupby(athlete).transform('max') - bw.groupby(athlete).transform('min'),
    'ath_first_age': both['Age'].groupby(athlete).transform('min'),
    'ath_n_same_year': both.groupby([athlete, both['MeetYear']])['Age'].transform('size').astype('float64'),
    'ath_share_tested': (both['Tested'] == 'Yes').astype('float64').groupby(athlete).transform('mean'),
    'ath_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation']),
    'ath0_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State']),
})
H2.loc[~valid] = np.nan

# E：比赛的经验结构
experience = H['ath_years_since_first']
E = pd.DataFrame({
    'meet_mean_ath_n': H['ath_n'].groupby(meet).transform('mean'),
    'meet_mean_ath_exp': experience.groupby(meet).transform('mean'),
    'meet_share_veterans': (experience >= 3).astype('float64').where(experience.notna()).groupby(meet).transform('mean'),
    'meet_share_newcomers': (H['ath_n'] == 1).astype('float64').where(H['ath_n'].notna()).groupby(meet).transform('mean'),
})

ROUND3 = {'H2': H2, 'E': E}
for name, frame in ROUND3.items():
    require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
    require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
NUMERIC_BLOCKS = {**BLOCKS, **BLOCKS2, **ROUND3}          # P T M（07）、M2 D H S（09）、H2 E（本轮）
R2_BLOCKS = ['P', 'T', 'M', 'H']                          # 09 的 ALL2
EXTRAS = {'lgb': [], 'xgb': ['M2', 'S', 'D'], 'catboost': ['M2', 'C']}   # PM：09 中各模型单独有效的块
VARIANTS3 = {'R2': [], 'H2': ['H2'], 'E': ['E'], 'PM': ['PM']}
SCREEN3 = [v for v in VARIANTS3 if RUN_ONLY is None or v in RUN_ONLY or v == 'R2']
_FRAMES3 = {}


def blocks_for(variant, model):
    out = list(R2_BLOCKS)
    for b in VARIANTS3[variant]:
        out += EXTRAS[model] if b == 'PM' else [b]
    return out


def runs(model, variant):
    """该模型在这个变体上的特征与 R2 不同才需要训练（例如 PM 对 LightGBM 没有变化）。"""
    return variant == 'R2' or blocks_for(variant, model) != blocks_for('R2', model)


def frames3(variant, model):
    """(train, test, 类别列)。"""
    blocks = blocks_for(variant, model)
    kind = 'str' if model == 'catboost' else 'category'
    key = (tuple(blocks), kind)
    if key not in _FRAMES3:
        nums = [both[RAW_NUM], F0_EXTRA] + [NUMERIC_BLOCKS[b] for b in blocks if b in NUMERIC_BLOCKS]
        cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
        frame = pd.concat(nums + cats, axis=1)
        require(frame.columns.is_unique, f'{variant} {model} 列名重复')
        cat_cols = [c for part in cats for c in part.columns]
        if kind == 'str':
            frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
        else:
            frame[cat_cols] = frame[cat_cols].astype('category')
        _FRAMES3[key] = (frame.iloc[:n_train].reset_index(drop=True),
                         frame.iloc[n_train:].reset_index(drop=True), cat_cols)
    return _FRAMES3[key]


new_cols = pd.concat([H2, E], axis=1)
display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                      'nan_share': new_cols.isna().mean()}).round(4))
print({f'{v}/{m}': frames3(v, m)[0].shape[1] for v in VARIANTS3 for m in MODELS if runs(m, v)})

# ---- 最终特征：10 的 ALL3（三个块都达标）----
_FRAMES3.clear()
VARIANTS3['ALL3'] = ['H2', 'E', 'PM']
FAMILY = {'lgb': 'lgb', 'lgb_rel': 'lgb', 'tabm': 'lgb', 'xgb': 'xgb', 'catboost': 'catboost', 'cb_rel': 'catboost'}
FRAMES = {m: frames3('ALL3', FAMILY[m]) for m in FINAL_MODELS}
for m, (Xtr_m, Xte_m, cats_m) in FRAMES.items():
    print(f'{m:9s} {Xtr_m.shape[1]} 列（{len(cats_m)} 个类别列），块：{blocks_for("ALL3", FAMILY[m])}')

,train_mean,test_mean,nan_share
ath_max_meet_states,7.2553,7.2965,0.0184
ath_mean_meet_states,3.2682,3.2702,0.0184
ath_max_meet_n,112.4144,112.6788,0.0184
ath_max_nat_states,17.7174,17.7553,0.0184
ath_ever_abroad,0.1208,0.1206,0.0184
ath_n_wclass,6.3779,6.4201,0.0184
ath_bw_range,48.6372,49.1032,0.0184
ath_first_age,24.1705,24.0925,0.0184
ath_n_same_year,4.9418,4.9522,0.0184
ath_share_tested,0.7628,0.7612,0.0184


{'R2/lgb': 41, 'R2/xgb': 41, 'R2/catboost': 41, 'H2/lgb': 53, 'H2/xgb': 53, 'H2/catboost': 53, 'E/lgb': 45, 'E/xgb': 45, 'E/catboost': 45, 'PM/xgb': 62, 'PM/catboost': 57}
lgb       57 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E']
xgb       78 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E', 'M2', 'S', 'D']
lgb_rel   57 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E']
catboost  73 列（15 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E', 'M2', 'C']
cb_rel    73 列（15 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E', 'M2', 'C']
tabm      57 列（13 个类别列），块：['P', 'T', 'M', 'H', 'H2', 'E']


## 6. 新模型的定义与预检

In [7]:
LGB_PARAMS = dict(learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, cat_smooth=20, verbose=-1)
XGB_PARAMS = dict(learning_rate=0.03, max_depth=8, min_child_weight=10.0, subsample=0.8, colsample_bytree=0.7,
                  reg_lambda=1.0, tree_method='hist', device='cuda', enable_categorical=True,
                  max_cat_to_onehot=1, eval_metric='rmse')
CB_PARAMS = dict(learning_rate=0.05, depth=8, l2_leaf_reg=10.0, bootstrap_type='Bernoulli', subsample=0.8,
                 border_count=128)
TABM_PARAMS = dict(device='cuda', n_cv=1, n_refit=0, verbosity=0, num_emb_type='pwl')
TABM_FALLBACK = dict(device='cuda', n_cv=1, n_refit=0, verbosity=0)
PATIENCE = {'lgb': 200, 'lgb_rel': 200, 'xgb': 200, 'catboost': 300, 'cb_rel': 300}
GBDT = [m for m in FINAL_MODELS if m != 'tabm']

SCALE = dots_scale(train['Sex'], train['BodyweightKg'])           # dots_scale 来自 09 的特征单元
SCALE_TEST = dots_scale(test['Sex'], test['BodyweightKg'])
require(min(SCALE.min(), SCALE_TEST.min()) > 0.3, 'DOTS 系数异常')

X_NN, XT_NN, CATS_NN = FRAMES['tabm']
NUM_NN = [c for c in X_NN.columns if c not in CATS_NN]
NAN_NN = [c for c in NUM_NN if X_NN[c].isna().any() or XT_NN[c].isna().any()]


def target_and_weight(model, rows):
    if model.endswith('_rel'):
        return y[rows] / SCALE[rows], SCALE[rows] ** 2       # 相对力量误差 × scale = kg 误差
    return y[rows], None


def gbdt_fit(model, seed, Xa, ya, wa, Xs=None, ys=None, ws=None, rounds=MAX_ROUNDS):
    """Xs 为 None 时不早停（全量重训）。返回 (模型, 最佳轮数)。"""
    cats = FRAMES[model][2]
    if model in ('lgb', 'lgb_rel'):
        m = lgb.LGBMRegressor(n_estimators=rounds, random_state=seed, n_jobs=-1, **LGB_PARAMS)
        if Xs is None:
            m.fit(Xa, ya, sample_weight=wa)
            return m, rounds
        m.fit(Xa, ya, sample_weight=wa, eval_set=[(Xs, ys)], eval_sample_weight=None if ws is None else [ws],
              callbacks=[lgb.early_stopping(PATIENCE[model], verbose=False)])
        return m, int(m.best_iteration_ or rounds)
    if model == 'xgb':
        if Xs is None:
            m = xgb.XGBRegressor(n_estimators=rounds, random_state=seed, **XGB_PARAMS)
            m.fit(Xa, ya, verbose=False)
            return m, rounds
        m = xgb.XGBRegressor(n_estimators=rounds, random_state=seed, early_stopping_rounds=PATIENCE['xgb'],
                             **XGB_PARAMS)
        m.fit(Xa, ya, eval_set=[(Xs, ys)], verbose=False)
        return m, int(m.best_iteration) + 1
    common = dict(iterations=rounds, loss_function='RMSE', eval_metric='RMSE', random_seed=seed, verbose=0,
                  task_type='GPU', devices='0', allow_writing_files=False, **CB_PARAMS)
    if Xs is None:
        m = catboost.CatBoostRegressor(**common)
        m.fit(catboost.Pool(Xa, ya, cat_features=cats, weight=wa))
        return m, rounds
    m = catboost.CatBoostRegressor(od_type='Iter', od_wait=PATIENCE[model], **common)
    m.fit(catboost.Pool(Xa, ya, cat_features=cats, weight=wa),
          eval_set=catboost.Pool(Xs, ys, cat_features=cats, weight=ws), use_best_model=True)
    return m, int(m.get_best_iteration()) + 1


def gbdt_predict(model, m, best, X):
    if model in ('lgb', 'lgb_rel'):
        return m.predict(X, num_iteration=best)
    if model == 'xgb':
        return m.predict(X, iteration_range=(0, best))
    return m.predict(X)


def nn_frames(*frames):
    """数值缺失用第一个表（外层训练部分）的中位数填补并加缺失标记；类别保持 category。"""
    median = frames[0][NUM_NN].median()
    out = []
    for frame in frames:
        z = frame.copy()
        for c in NAN_NN:
            z[f'{c}_isna'] = z[c].isna().astype('float64')
        z[NUM_NN] = z[NUM_NN].fillna(median).fillna(0.0)
        out.append(z.reset_index(drop=True))
    return out


def tabm_fit_predict(seed, a, s, v, Xt, overrides=None):
    require(TabM_D_Regressor is not None, f'pytabkit 不可用：{PYTABKIT_ERROR}')
    params = dict(TABM_PARAMS, random_state=seed, **(overrides or {}))
    unknown = set(params) - set(inspect.signature(TabM_D_Regressor.__init__).parameters)
    require(not unknown, f'TabM 不支持参数 {sorted(unknown)}')
    Za, Zs, Zv, Zt = nn_frames(X_NN.iloc[a], X_NN.iloc[s], X_NN.iloc[v], Xt)
    mu, sd = y[a].mean(), y[a].std()
    m = TabM_D_Regressor(**params)
    m.fit(Za, (y[a] - mu) / sd, Zs, (y[s] - mu) / sd, cat_col_names=[c for c in Za.columns if c in CATS_NN])
    p_val = np.asarray(m.predict(Zv), dtype='float64').reshape(-1) * sd + mu
    p_test = np.asarray(m.predict(Zt), dtype='float64').reshape(-1) * sd + mu
    del m
    gc.collect()
    torch.cuda.empty_cache()
    return p_val, p_test


def fit_fold(model, seed, a, s, v, test_rows=None, rounds=MAX_ROUNDS, overrides=None):
    """a / s / v：内层拟合 / 内层早停 / 外层验证 的训练行号。返回 (验证预测, 测试预测, 最佳轮数或 None)。"""
    Xtr, Xte, _ = FRAMES[model]
    Xt = Xte if test_rows is None else Xte.iloc[test_rows]
    scale_t = SCALE_TEST if test_rows is None else SCALE_TEST[test_rows]
    if model == 'tabm':
        p_val, p_test = tabm_fit_predict(seed, a, s, v, Xt, overrides)
        return p_val, p_test, None
    ya, wa = target_and_weight(model, a)
    ys, ws = target_and_weight(model, s)
    if wa is not None:
        norm = wa.mean()
        wa, ws = wa / norm, ws / norm
    m, best = gbdt_fit(model, seed, Xtr.iloc[a], ya, wa, Xtr.iloc[s], ys, ws, rounds)
    p_val, p_test = gbdt_predict(model, m, best, Xtr.iloc[v]), gbdt_predict(model, m, best, Xt)
    if model.endswith('_rel'):
        p_val, p_test = np.asarray(p_val) * SCALE[v], np.asarray(p_test) * scale_t
    return np.asarray(p_val, dtype='float64'), np.asarray(p_test, dtype='float64'), best


def fit_full(model, seed, rounds, test_rows=None):
    """全部训练行、固定轮数、不早停。返回测试预测。"""
    Xtr, Xte, _ = FRAMES[model]
    Xt = Xte if test_rows is None else Xte.iloc[test_rows]
    scale_t = SCALE_TEST if test_rows is None else SCALE_TEST[test_rows]
    rows = np.arange(n_train)
    ya, wa = target_and_weight(model, rows)
    if wa is not None:
        wa = wa / wa.mean()
    m, _ = gbdt_fit(model, seed, Xtr, ya, wa, rounds=rounds)
    p_test = gbdt_predict(model, m, rounds, Xt)
    if model.endswith('_rel'):
        p_test = np.asarray(p_test) * scale_t
    return np.asarray(p_test, dtype='float64')



from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

X_BASE, XT_BASE, CATS = FRAMES['lgb']            # R2 + H2 + E，类别为 category（与 TabM 相同）
NUMS = [c for c in X_BASE.columns if c not in CATS]
PYTABKIT_CLASSES = {'mlp_plr': 'MLP_PLR_D_Regressor', 'resnet': 'Resnet_RTDL_D_Regressor',
                    'ftt': 'FTT_D_Regressor', 'tabr': 'TabR_S_D_Regressor'}


def numeric_with_codes(enc, X, fill=None):
    num = X[NUMS].to_numpy(dtype='float64')
    if fill is not None:
        num = np.where(np.isnan(num), fill, num)
    return np.hstack([num, enc.transform(X[CATS])])


def fit_ridge(seed, a, s, v, Xt, small=False):
    rows = np.concatenate([a, s])
    pre = ColumnTransformer([
        ('num', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), NUMS),
        ('cat', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=20), CATS)])
    model = make_pipeline(pre, Ridge(alpha=10.0)).fit(X_BASE.iloc[rows], y[rows])
    return model.predict(X_BASE.iloc[v]), model.predict(Xt)


def fit_hgb(seed, a, s, v, Xt, small=False):
    rows = np.concatenate([a, s])
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan, encoded_missing_value=np.nan,
                         max_categories=250).fit(X_BASE.iloc[rows][CATS])
    mask = [False] * len(NUMS) + [True] * len(CATS)
    model = HistGradientBoostingRegressor(
        categorical_features=mask, learning_rate=0.05, max_iter=20 if small else 3000, max_leaf_nodes=63,
        min_samples_leaf=40, l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=100, random_state=seed)
    model.fit(numeric_with_codes(enc, X_BASE.iloc[rows]), y[rows])           # 内部留 10% 早停
    return model.predict(numeric_with_codes(enc, X_BASE.iloc[v])), model.predict(numeric_with_codes(enc, Xt))


def fit_forest(kind, seed, a, s, v, Xt, small=False):
    rows = np.concatenate([a, s])
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit(X_BASE.iloc[rows][CATS])
    common = dict(n_estimators=10 if small else 200, min_samples_leaf=5, n_jobs=-1, random_state=seed)
    model = (RandomForestRegressor(max_features=0.3, max_samples=0.5, **common) if kind == 'rf'
             else ExtraTreesRegressor(max_features=0.5, **common))
    model.fit(numeric_with_codes(enc, X_BASE.iloc[rows], fill=-999.0), y[rows])
    return (model.predict(numeric_with_codes(enc, X_BASE.iloc[v], fill=-999.0)),
            model.predict(numeric_with_codes(enc, Xt, fill=-999.0)))


def fit_ebm(seed, a, s, v, Xt, small=False):
    from interpret.glassbox import ExplainableBoostingRegressor
    rows = np.concatenate([a, s])
    model = ExplainableBoostingRegressor(interactions=0 if small else 5, outer_bags=1 if small else 2,
                                         max_rounds=50 if small else 5000, n_jobs=-1, random_state=seed)
    model.fit(X_BASE.iloc[rows], y[rows])                                   # 内部留出验证集早停
    return np.asarray(model.predict(X_BASE.iloc[v])), np.asarray(model.predict(Xt))


def make_tabpfn():
    from tabpfn import TabPFNRegressor
    if TABPFN_TOKEN:
        return TabPFNRegressor(device='cuda', ignore_pretraining_limits=True)
    from tabpfn.constants import ModelVersion
    return TabPFNRegressor.create_default_for_version(ModelVersion.V2)


def fit_tabpfn(seed, a, s, v, Xt, small=False):
    rows = np.concatenate([a, s])
    context = np.random.default_rng(seed).choice(rows, size=min(500 if small else TABPFN_CONTEXT_ROWS, len(rows)),
                                                 replace=False)
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan,
                         encoded_missing_value=np.nan).fit(X_BASE.iloc[context][CATS])
    model = make_tabpfn()
    model.fit(numeric_with_codes(enc, X_BASE.iloc[context]), y[context])
    p_val = np.asarray(model.predict(numeric_with_codes(enc, X_BASE.iloc[v])), dtype='float64')
    p_test = np.asarray(model.predict(numeric_with_codes(enc, Xt)), dtype='float64')
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return p_val, p_test


def fit_pytabkit(name, seed, a, s, v, Xt, small=False):
    if name == 'tabr':
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'], check=False)
    import pytabkit
    cls = getattr(pytabkit, PYTABKIT_CLASSES[name])
    params = dict(device='cuda', random_state=seed, n_cv=1, n_refit=0, verbosity=0)
    if small:
        params['max_epochs'] = 2
    unknown = set(params) - set(inspect.signature(cls.__init__).parameters)
    require(not unknown, f'{name} 不支持参数 {sorted(unknown)}')
    Za, Zs, Zv, Zt = nn_frames(X_NN.iloc[a], X_NN.iloc[s], X_NN.iloc[v], Xt)    # 与 TabM 相同的填补
    mu, sd = y[a].mean(), y[a].std()
    m = cls(**params)
    m.fit(Za, (y[a] - mu) / sd, Zs, (y[s] - mu) / sd, cat_col_names=[c for c in Za.columns if c in CATS_NN])
    p_val = np.asarray(m.predict(Zv), dtype='float64').reshape(-1) * sd + mu
    p_test = np.asarray(m.predict(Zt), dtype='float64').reshape(-1) * sd + mu
    del m
    gc.collect()
    torch.cuda.empty_cache()
    return p_val, p_test


def fit_new(name, seed, a, s, v, Xt, small=False):
    if name == 'ridge':
        return fit_ridge(seed, a, s, v, Xt, small)
    if name == 'hgb':
        return fit_hgb(seed, a, s, v, Xt, small)
    if name in ('rf', 'et'):
        return fit_forest(name, seed, a, s, v, Xt, small)
    if name == 'ebm':
        return fit_ebm(seed, a, s, v, Xt, small)
    if name == 'tabpfn':
        return fit_tabpfn(seed, a, s, v, Xt, small)
    return fit_pytabkit(name, seed, a, s, v, Xt, small)


def save_result(stem, arrays, info):
    np.savez(OUTPUT_DIR / f'{stem}.npz', **arrays)
    info = {**info, 'gpu': GPU_NAME, 'versions': VERSIONS, 'finished_at_utc': datetime.now(timezone.utc).isoformat()}
    (OUTPUT_DIR / f'{stem}.json').write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding='utf-8')
    return info


# 预检：每个模型在 3,000 行上用很小的预算跑一次（几秒到一两分钟），失败的自动跳过
rng = np.random.default_rng(SEED)
probe = rng.choice(n_train, 3000, replace=False)
pa, ps, pv = probe[:2000], probe[2000:2500], probe[2500:]
failed = {}
for name in COMPARE + OPTIONAL:
    started = time.perf_counter()
    try:
        p_val, p_test = fit_new(name, SEED, pa, ps, pv, XT_BASE.iloc[:300], small=True)
        p_val, p_test = np.asarray(p_val, dtype='float64'), np.asarray(p_test, dtype='float64')
        require(len(p_val) == len(pv) and len(p_test) == 300, '预测长度不符')
        require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), '预测含非有限值')
        print(f'预检通过：{name:8s} RMSE（小预算）{rmse(y[pv], p_val):.1f}，用时 {time.perf_counter() - started:.0f}s')
    except Exception as exc:
        failed[name] = repr(exc)
        print(f'预检失败，跳过：{name} → {exc!r}')
RUN_NEW = [n for n in COMPARE + OPTIONAL if n not in failed]
print('将运行：', RUN_NEW)

预检通过：ridge    RMSE（小预算）85.4，用时 0s
预检通过：hgb      RMSE（小预算）101.6，用时 1s
预检通过：et       RMSE（小预算）87.3，用时 0s
预检通过：rf       RMSE（小预算）88.7，用时 0s


tabpfn-v2-regressor.ckpt: reconstructing file:   0%|          |  0.00B / 44.4MB            

tabpfn-v2-regressor.ckpt: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

预检通过：tabpfn   RMSE（小预算）81.4，用时 12s


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


预检失败，跳过：mlp_plr → ModuleNotFoundError("No module named 'skorch'")


/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


预检失败，跳过：resnet → ModuleNotFoundError("No module named 'skorch'")
预检通过：ebm      RMSE（小预算）90.9，用时 2s
将运行： ['ridge', 'hgb', 'et', 'rf', 'tabpfn', 'ebm']


## 7. 五折训练（按模型逐个跑完，可续跑）

In [8]:
def run_new(name, fold):
    stem = f'{name}__s{SEED}__fold{fold}'
    json_path, npz_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.npz'
    if json_path.is_file() and npz_path.is_file():
        return json.loads(json_path.read_text(encoding='utf-8')), 'cache'
    fit_idx, valid_idx = np.flatnonzero(FOLD_IDS != fold), np.flatnonzero(FOLD_IDS == fold)
    inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED)  # = 11
    started = time.perf_counter()
    p_val, p_test = fit_new(name, SEED, inner_fit, inner_stop, valid_idx, XT_BASE)
    p_val, p_test = np.asarray(p_val, dtype='float64'), np.asarray(p_test, dtype='float64')
    require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
    info = save_result(stem, {'valid_idx': valid_idx, 'val': p_val, 'test': p_test},
                       {'model': name, 'fold': fold, 'rmse': rmse(y[valid_idx], p_val),
                        'seconds': time.perf_counter() - started})
    return info, f'{info["seconds"]:.0f}s'


for name in RUN_NEW:
    started = time.perf_counter()
    try:
        for fold in range(1, N_FOLDS + 1):
            info, how = run_new(name, fold)
            print(f'{name:8s} fold {fold}  RMSE {info["rmse"]:.3f}  ({how})')
        print(f'—— {name} 完成，本次运行用时 {(time.perf_counter() - started) / 60:.1f} 分钟')
    except Exception as exc:
        print(f'!! {name} 运行出错，跳过剩下的折：{exc!r}')

ridge    fold 1  RMSE 84.839  (8s)
ridge    fold 2  RMSE 84.636  (8s)
ridge    fold 3  RMSE 85.784  (8s)
ridge    fold 4  RMSE 85.209  (8s)
ridge    fold 5  RMSE 84.369  (8s)
—— ridge 完成，本次运行用时 0.6 分钟
hgb      fold 1  RMSE 77.724  (40s)
hgb      fold 2  RMSE 78.102  (27s)
hgb      fold 3  RMSE 78.288  (29s)
hgb      fold 4  RMSE 78.216  (34s)
hgb      fold 5  RMSE 77.070  (31s)
—— hgb 完成，本次运行用时 2.7 分钟
et       fold 1  RMSE 80.933  (57s)
et       fold 2  RMSE 81.266  (56s)
et       fold 3  RMSE 81.586  (56s)
et       fold 4  RMSE 81.088  (55s)
et       fold 5  RMSE 80.394  (56s)
—— et 完成，本次运行用时 4.7 分钟
rf       fold 1  RMSE 81.549  (65s)
rf       fold 2  RMSE 81.730  (65s)
rf       fold 3  RMSE 82.142  (65s)
rf       fold 4  RMSE 81.594  (65s)
rf       fold 5  RMSE 81.060  (65s)
—— rf 完成，本次运行用时 5.4 分钟
tabpfn   fold 1  RMSE 81.370  (572s)
tabpfn   fold 2  RMSE 81.636  (571s)
tabpfn   fold 3  RMSE 82.162  (575s)
tabpfn   fold 4  RMSE 81.625  (576s)
tabpfn   fold 5  RMSE 81.108  (571s)
—— t

## 8. 比较：单独的分数（种子 42）与对最终融合的贡献（五折嵌套 NNLS）

In [9]:
def load_folds(directory, stem_prefix):
    oof, test_pred = np.full(n_train, np.nan), np.zeros(len(test))
    for fold in range(1, N_FOLDS + 1):
        path = directory / f'{stem_prefix}__fold{fold}.npz'
        if not path.is_file():
            return None
        saved = np.load(path)
        oof[saved['valid_idx']] = saved['val']
        test_pred += saved['test'] / N_FOLDS
    return oof, test_pred


def fold_rmse(pred):
    return np.array([rmse(y[FOLD_IDS == k], pred[FOLD_IDS == k]) for k in range(1, N_FOLDS + 1)])


def nested_nnls(P):
    out = np.zeros(n_train)
    for k in range(1, N_FOLDS + 1):
        fit, va = FOLD_IDS != k, FOLD_IDS == k
        w, _ = nnls(P[fit], y[fit])
        out[va] = P[va] @ w
    return out


# 11 的 6 个模型：种子 42 的 OOF（单独比较用）、三种子平均 OOF（融合用）、最好的测试预测
BASE = {}
for model in FINAL_MODELS:
    per_seed = {s: load_folds(PREV_DIR, f'{model}__s{s}') for s in PREV_SEEDS}
    require(all(v is not None for v in per_seed.values()), f'11 缺少 {model} 的结果')
    fulls = [np.load(p)['test'] for d in (PREV_DIR, BOOST_DIR) if d.is_dir() for p in sorted(d.glob(f'{model}__s*__full.npz'))]
    near = [np.load(p)['test'] for p in sorted(BOOST_DIR.glob('tabm__s*__nearfull.npz'))] if BOOST_DIR.is_dir() else []
    fold_avg = np.mean([t for _, t in per_seed.values()], axis=0)
    best_test = np.mean(fulls, axis=0) if fulls else (np.mean(near, axis=0) if (model == 'tabm' and near) else fold_avg)
    BASE[model] = {'oof42': per_seed[42][0], 'oof': np.mean([o for o, _ in per_seed.values()], axis=0),
                   'test': best_test}
NEW = {}
for name in COMPARE + OPTIONAL:
    loaded = load_folds(OUTPUT_DIR, f'{name}__s{SEED}')
    if loaded is not None:
        NEW[name] = {'oof42': loaded[0], 'oof': loaded[0], 'test': loaded[1]}

base_names = list(BASE)
P_base = np.column_stack([BASE[m]['oof'] for m in base_names])
base_nested = nested_nnls(P_base)
base_folds = fold_rmse(base_nested)
rows = []
for name, item in {**BASE, **NEW}.items():
    row = {'model': name, '来源': '11' if name in BASE else '13', 'oof_seed42': rmse(y, item['oof42']),
           'corr_with_blend': float(np.corrcoef(item['oof42'], base_nested)[0, 1])}
    if name in NEW:
        P = np.column_stack([P_base, item['oof']])
        nested = nested_nnls(P)
        delta = fold_rmse(nested) - base_folds
        w, _ = nnls(P, y)
        row.update({'blend_nested': rmse(y, nested), 'delta_vs_blend': rmse(y, nested) - rmse(y, base_nested),
                    'folds_better': int((delta < 0).sum()), 'weight_in_blend': float(w[-1])})
    rows.append(row)
compare = pd.DataFrame(rows).set_index('model').sort_values('oof_seed42')
print(f'11 的 6 模型融合（嵌套）：{rmse(y, base_nested):.3f}')
display(compare.round(3))
compare.to_csv(OUTPUT_DIR / 'summary_compare.csv')
seconds = {}
for p in OUTPUT_DIR.glob('*__fold*.json'):
    info = json.loads(p.read_text(encoding='utf-8'))
    seconds.setdefault(info['model'], []).append(info['seconds'])
print('每折平均用时（秒）：', {k: round(float(np.mean(v))) for k, v in seconds.items()})

KEEP_NEW = [n for n in NEW if compare.loc[n, 'delta_vs_blend'] <= ADD_DELTA and compare.loc[n, 'folds_better'] >= 4]
print('达到加入门槛的新模型：', KEEP_NEW or '无')

11 的 6 模型融合（嵌套）：76.395


,来源,oof_seed42,corr_with_blend,blend_nested,delta_vs_blend,folds_better,weight_in_blend
model,,,,,,,
cb_rel,11,77.167,0.997,NaN,NaN,NaN,NaN
tabm,11,77.493,0.996,NaN,NaN,NaN,NaN
catboost,11,77.497,0.996,NaN,NaN,NaN,NaN
lgb_rel,11,77.646,0.996,NaN,NaN,NaN,NaN
lgb,11,77.709,0.995,NaN,NaN,NaN,NaN
hgb,13,77.881,0.994,76.361,-0.034,5.0,0.147
xgb,11,78.458,0.993,NaN,NaN,NaN,NaN
et,13,81.054,0.985,76.395,0.000,0.0,0.000
ebm,13,81.448,0.985,76.395,0.000,0.0,0.000


每折平均用时（秒）： {'ridge': 8, 'hgb': 32, 'et': 56, 'rf': 65, 'tabpfn': 573, 'ebm': 88}
达到加入门槛的新模型： ['hgb']


## 9. 如有达标的新模型：加入融合 → 双胞胎 + S3 → 导出提交文件（与 11 相同的第二层）

In [10]:
if KEEP_NEW:
    names = base_names + KEEP_NEW
    ITEMS = {**BASE, **{n: NEW[n] for n in KEEP_NEW}}
    P_all = np.column_stack([ITEMS[n]['oof'] for n in names])
    WEIGHTS_PLUS, _ = nnls(P_all, y)
    NESTED = nested_nnls(P_all)
    BLEND_TEST = {'v3_plus': np.column_stack([ITEMS[n]['test'] for n in names]) @ WEIGHTS_PLUS}
    print('权重：', dict(zip(names, WEIGHTS_PLUS.round(3))))
    print(f'嵌套融合：{rmse(y, base_nested):.3f} → {rmse(y, NESTED):.3f}')
else:
    NESTED, BLEND_TEST = None, None
    print('没有新模型达到门槛，最终提交保持 11 / 12 的版本。')

权重： {'lgb': np.float64(0.0), 'xgb': np.float64(0.152), 'lgb_rel': np.float64(0.009), 'catboost': np.float64(0.033), 'cb_rel': np.float64(0.454), 'tabm': np.float64(0.205), 'hgb': np.float64(0.147)}
嵌套融合：76.395 → 76.361


In [11]:
if BLEND_TEST is not None:
    CORE = CORE9


    def row_keys(frame, columns):
        return pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()


    def mate_stats(train_keys, query_keys=None):
        table = pd.DataFrame({'k': train_keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
        if query_keys is None:                      # 训练行：留一
            keys = pd.Series(train_keys)
            n = keys.map(table['count']).to_numpy() - 1
            s = keys.map(table['sum']).to_numpy() - y
        else:                                       # 测试行：全部训练行
            keys = pd.Series(query_keys)
            n = keys.map(table['count']).fillna(0).to_numpy()
            s = keys.map(table['sum']).fillna(0).to_numpy()
        return n, np.where(n > 0, s / np.maximum(n, 1), np.nan)


    def twins(query_exact=None, query_core=None):
        n_ex, m_ex = mate_stats(row_keys(train, RAW_FEATURES), query_exact)
        n_co, m_co = mate_stats(row_keys(train, CORE), query_core)
        level = np.select([n_ex > 0, n_co > 0], ['exact', 'core'], default='none')
        return level, np.where(level == 'exact', m_ex, m_co)


    def fit_alpha(mask, pred, mate):
        d = mate[mask] - pred[mask]
        denom = float(np.sum(d * d))
        return float(np.clip(np.sum((y[mask] - pred[mask]) * d) / denom, 0, 1)) if denom > 0 else 0.0


    def apply_twins(pred, mate, level, alphas):
        out = pred.copy()
        for lv, a in alphas.items():
            m = level == lv
            out[m] = pred[m] + a * (mate[m] - pred[m])
        return out


    LEVEL, MATE = twins()
    T_LEVEL, T_MATE = twins(row_keys(test, RAW_FEATURES), row_keys(test, CORE))
    HALF = (pd.util.hash_pandas_object(train[['Sex', 'Country', 'State']], index=False).to_numpy() % 2).astype(int)
    TR, TE = train.reset_index(drop=True), test.reset_index(drop=True)


    def text_key(frame, columns):
        out = frame[columns[0]].astype(str)
        for c in columns[1:]:
            out = out + '|' + frame[c].astype(str)
        return out


    BY_TR = np.floor(TR['MeetYear'] - TR['Age'] + 1e-9).astype(int)
    BY_TE = np.floor(TE['MeetYear'] - TE['Age'] + 1e-9).astype(int)
    YR_TR, YR_TE = TR['MeetYear'].astype(int), TE['MeetYear'].astype(int)
    PREFIX = {'A0': ['Sex', 'Country', 'State'], 'A1': ['Sex', 'Country', 'State', 'Federation'],
              'A2': ['Sex', 'Country', 'State', 'Federation', 'Equipment'],
              'A3': ['Sex', 'Country', 'State', 'Federation', 'Equipment', 'WeightClassKg']}
    SHRINK = {'A0': 5.0, 'A1': 5.0, 'A2': 3.0, 'A3': 2.0}
    PFX_TR = {k: text_key(TR, c) for k, c in PREFIX.items()}
    PFX_TE = {k: text_key(TE, c) for k, c in PREFIX.items()}
    MEET_COLS = ['Federation', 'MeetYear', 'MeetCountry', 'MeetState', 'Sex', 'Equipment']
    MEET_TR, MEET_TE = text_key(TR, MEET_COLS), text_key(TE, MEET_COLS)
    VALID_TR = (TR['Country'] != 'Unknown').to_numpy(dtype='float64')
    VALID_TE = (TE['Country'] != 'Unknown').to_numpy(dtype='float64')


    def sums(keys, values):
        table = pd.DataFrame({'k': keys.to_numpy(), 'v': values}).groupby('k')['v'].agg(['sum', 'count'])
        return table['sum'], table['count']


    def look(table, keys):
        s, n = table
        return keys.map(s).fillna(0).to_numpy(dtype='float64'), keys.map(n).fillna(0).to_numpy(dtype='float64')


    def f_exact(k, r):
        ktr, kte = PFX_TR[k] + '|' + BY_TR.astype(str), PFX_TE[k] + '|' + BY_TE.astype(str)
        table = sums(ktr, r)
        s, n = look(table, ktr)
        st, nt = look(table, kte)
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_fuzzy(k, r):
        table = sums(PFX_TR[k] + '|' + BY_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in (-1, 0, 1):
            a, b = look(table, PFX_TR[k] + '|' + (BY_TR + d).astype(str))
            s, n = s + a, n + b
            a, b = look(table, PFX_TE[k] + '|' + (BY_TE + d).astype(str))
            st, nt = st + a, nt + b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_decay(k, r, tau=3.0, max_gap=8):
        base_tr = PFX_TR[k] + '|' + BY_TR.astype(str) + '|'
        base_te = PFX_TE[k] + '|' + BY_TE.astype(str) + '|'
        table = sums(base_tr + YR_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in range(-max_gap, max_gap + 1):
            w = float(np.exp(-abs(d) / tau))
            a, b = look(table, base_tr + (YR_TR + d).astype(str))
            s, n = s + w * a, n + w * b
            a, b = look(table, base_te + (YR_TE + d).astype(str))
            st, nt = st + w * a, nt + w * b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])     # d = 0 时自己权重为 1，已扣除


    def f_meet(r, lam=20.0):
        table = sums(MEET_TR, r)
        s, n = look(table, MEET_TR)
        st, nt = look(table, MEET_TE)
        return (s - r) / (n - 1 + lam), st / (nt + lam)


    def s3_features(r):
        cols_tr, cols_te = [], []
        for builder in (f_exact, f_fuzzy, f_decay):
            for k in PREFIX:
                a, b = builder(k, r)
                cols_tr.append(a * VALID_TR)
                cols_te.append(b * VALID_TE)
        a, b = f_meet(r)
        cols_tr.append(a)
        cols_te.append(b)
        return np.column_stack(cols_tr), np.column_stack(cols_te)


    # 第二层只依赖训练行的 OOF（嵌套融合），对三种测试预测使用同一组 alpha / 系数
    alphas = {lv: fit_alpha(LEVEL == lv, NESTED, MATE) for lv in ['exact', 'core']}
    p1 = apply_twins(NESTED, MATE, LEVEL, alphas)
    p1_cf = NESTED.copy()
    for h in (0, 1):
        a_h = {lv: fit_alpha((HALF != h) & (LEVEL == lv), NESTED, MATE) for lv in ['exact', 'core']}
        m = HALF == h
        p1_cf[m] = apply_twins(NESTED[m], MATE[m], LEVEL[m], a_h)
    r1 = y - p1
    F, FT = s3_features(r1)
    p2_cf = p1_cf.copy()
    for h in (0, 1):
        fit, m = HALF != h, HALF == h
        beta_h = np.linalg.lstsq(F[fit], r1[fit], rcond=None)[0]
        p2_cf[m] = p1_cf[m] + F[m] @ beta_h
    BETA = np.linalg.lstsq(F, r1, rcond=None)[0]
    STAGE2_ADJ = FT @ BETA
    stage_table = pd.DataFrame([{'step': '融合（嵌套）', 'oof_rmse': rmse(y, NESTED)},
                                {'step': '+ 双胞胎（交叉拟合）', 'oof_rmse': rmse(y, p1_cf)},
                                {'step': '+ S3 平滑（交叉拟合）', 'oof_rmse': rmse(y, p2_cf)}]).set_index('step')
    display(stage_table.round(3))
    stage_table.to_csv(OUTPUT_DIR / 'summary_stage2.csv')
    print('alpha：', {k: round(v, 3) for k, v in alphas.items()}, '｜测试中有双胞胎的行：', int((T_LEVEL != 'none').sum()),
          '｜S3 平均调整幅度：', round(float(np.abs(STAGE2_ADJ).mean()), 2), 'kg')

    for kind, pred in BLEND_TEST.items():
        final = apply_twins(pred, T_MATE, T_LEVEL, alphas) + STAGE2_ADJ
        for stage, values in [('base', pred), ('final', final)]:
            path = OUTPUT_DIR / f'submission_final_{kind}_{stage}.csv'
            pd.DataFrame({'id': test['id'], TARGET: np.clip(values, 0, None)}).to_csv(path, index=False)
            print('写出', path.name)
    np.save(OUTPUT_DIR / 'oof_nested_blend.npy', NESTED)
    np.save(OUTPUT_DIR / 'oof_final_crossfit.npy', p2_cf)
print('完成。跑完告诉 Claude，本地会同步 OUTPUT_DIR 的结果。')

,oof_rmse
step,
融合（嵌套）,76.361
+ 双胞胎（交叉拟合）,76.138
+ S3 平滑（交叉拟合）,76.010


alpha： {'exact': 0.854, 'core': 0.52} ｜测试中有双胞胎的行： 519 ｜S3 平均调整幅度： 3.7 kg
写出 submission_final_v3_plus_base.csv
写出 submission_final_v3_plus_final.csv
完成。跑完告诉 Claude，本地会同步 OUTPUT_DIR 的结果。
